In [12]:
import BigDataAnalysis_Package_RF as bda
import os
from joblib import Parallel, delayed
import pandas as pd

import importlib
importlib.reload(bda)

<module 'BigDataAnalysis_Package_RF' from 'd:\\Code\\Python\\BigDataAnalysis_Package_RF.py'>

In [ ]:
X_a_path = 'Data/Xa'
Y_a_path = 'Data/Ya'
X_a = bda.read_data(X_a_path)
Y_a = bda.read_data(Y_a_path)
X_a = bda.tune_data(X_a, 15000)
Y_a = bda.tune_data(Y_a, 15000)
Data = {'Xa':X_a, 'Ya':Y_a}
all_dict = {}
for _, d in Data.items():
    all_dict.update(d)

In [3]:
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

centers_bands = {}
for group in ("Xa", "Ya"):
    centers_bands[group] = bda.build_centers_bands_for_group(all_dict, group, fmax=1250)

items = list(all_dict.items())
key_cols = ["Direction", "Load", "Index"]

base_chunk_size = 30  
base_chunks = list(bda.chunk_list(items, base_chunk_size))

base_rows_nested = Parallel(n_jobs=-1,backend="loky",batch_size=1,pre_dispatch="2*n_jobs")(delayed(bda._base_chunk)(ch) for ch in base_chunks)

base_rows = [r for chunk in base_rows_nested for r in chunk]
base_df = pd.DataFrame(base_rows)

base_df["GT_Health"] = base_df.apply(bda.assign_gt_health, axis=1)
base_df = base_df.set_index(key_cols, drop=False)

chunk_size = 10
chunks = list(bda.chunk_list(items, chunk_size))
abn_rows_nested = Parallel(n_jobs=-1, backend="loky", batch_size=1, pre_dispatch="2*n_jobs")(delayed(bda._abn_chunk)(ch, centers_bands) for ch in chunks)

abn_rows = [r for chunk in abn_rows_nested for r in chunk]

abn_df = pd.DataFrame(abn_rows)
abn_df = abn_df.set_index(key_cols, drop=False)

In [4]:
train_pack = {}
for g in ('Xa', 'Ya'):
    train_pack[g] = bda.build_train_pack(base_df, abn_df, g)


In [ ]:
out_dir = 'final_models_rf'
bda.finalize_and_save(train_pack["Xa"], "Xa", out_dir, centers_bands_group=centers_bands["Xa"])
bda.finalize_and_save(train_pack["Ya"], "Ya", out_dir, centers_bands_group=centers_bands["Ya"])

In [ ]:
param_path = 'final_models_rf'
Xa_test_path = 'Data/test/Xa'
Ya_test_path = 'Data/test/Ya'
Xa_test_single_path = 'Data/test/Xa/gvb000001.txt'

In [51]:
bda.predict_by_rf(Xa_test_path, param_path)

,HI_mean,decision,HI_65,HI_95,HI_130
data_root_gvb000001.txt,4.7,Abmormal,NaN,NaN,NaN
data_root_gvb000002.txt,45.8,Normal,100.0,76.3,64.2
data_root_gvb000003.txt,0.2,Abmormal,NaN,NaN,NaN
data_root_gvb000004.txt,9.5,Normal,79.5,62.2,37.4
data_root_gvb000005.txt,8.7,Abnormal,23.9,74.1,34.3
...,...,...,...,...,...
data_root_gvb000076.txt,24.0,Normal,100.0,82.7,50.5
data_root_gvb000077.txt,0.3,Abmormal,NaN,NaN,NaN
data_root_gvb000078.txt,98.7,Normal,NaN,NaN,NaN
data_root_gvb000079.txt,99.6,Normal,NaN,NaN,NaN


In [16]:
bda.predict_by_rf(Xa_test_single_path , param_path, fold = False)

,HI_mean,decision
gvb000001,4.7,Abnormal


In [53]:
bda.predict_by_rf(Ya_test_path, param_path, group_prefix = "Y")

,HI_mean,decision
data_root_gvb000001.txt,100.0,Normal
data_root_gvb000002.txt,0.0,Abmormal
data_root_gvb000003.txt,0.0,Abmormal
data_root_gvb000004.txt,0.0,Abmormal
data_root_gvb000005.txt,0.2,Abmormal
...,...,...
data_root_gvb000076.txt,0.0,Abmormal
data_root_gvb000077.txt,100.0,Normal
data_root_gvb000078.txt,0.0,Abmormal
data_root_gvb000079.txt,0.0,Abmormal
